# 01 - Profile the HVFHV source before designing any metric

**Operational question:** where does pickup delay accumulate in the NYC ride lifecycle, and how does it vary by
location, time and weather?

This notebook is **exploration only**. It answers: what does the source actually contain, what is the grain,
how complete are the lifecycle events, what quality problems exist, and which metrics are defensible?
The repeatable logic lives in `src/` and runs via `python run_pipeline.py --month 2026-07`.

Rules carried over from FlashEats: profile before defining KPIs; a populated column is not an observed event;
quantify problems instead of silently fixing them; association is not causation.

In [1]:
from pathlib import Path
import json
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MONTH = "2026-07"
RAW = ROOT / "data" / "raw" / "hvfhv" / MONTH / f"fhvhv_tripdata_{MONTH}.parquet"
print("raw file:", RAW.relative_to(ROOT), f"{RAW.stat().st_size / 1e6:,.0f} MB")
print("manifest:", json.loads((RAW.parent / "_retrieval_manifest.json").read_text())["sha256"][:16], "...")

raw file: data\raw\hvfhv\2026-07\fhvhv_tripdata_2026-07.parquet 511 MB
manifest: 8da280d9d23813aa ...


## 1. Schema, row count, grain
Read the Parquet footer first: it is cheap and tells us what we are dealing with.

In [2]:
pf = pq.ParquetFile(RAW)
print(f"rows={pf.metadata.num_rows:,} row_groups={pf.metadata.num_row_groups}")
print(pf.schema_arrow)

rows=20,921,249 row_groups=20
hvfhs_license_num: large_string
dispatching_base_num: large_string
originating_base_num: large_string
request_datetime: timestamp[us]
on_scene_datetime: timestamp[us]
pickup_datetime: timestamp[us]
dropoff_datetime: timestamp[us]
PULocationID: int32
DOLocationID: int32
trip_miles: double
trip_time: int64
base_passenger_fare: double
tolls: double
bcf: double
sales_tax: double
congestion_surcharge: double
airport_fee: double
tips: double
driver_pay: double
shared_request_flag: large_string
shared_match_flag: large_string
access_a_ride_flag: large_string
wav_request_flag: large_string
wav_match_flag: large_string
cbd_congestion_fee: double


**Observation:** there is **no trip identifier**. One row appears to be one completed trip (it has one pickup and
one dropoff). Grain has to be *tested*, not assumed. Section 7 checks that the full natural key is unique.

In [3]:
cols = ["hvfhs_license_num", "request_datetime", "on_scene_datetime", "pickup_datetime", "dropoff_datetime",
        "PULocationID", "DOLocationID", "trip_miles", "trip_time", "base_passenger_fare"]
t = pq.read_table(RAW, columns=cols)
n = t.num_rows
def pct(mask): return f"{pc.sum(pc.cast(mask, pa.int64())).as_py():,} ({pc.sum(pc.cast(mask, pa.int64())).as_py() / n:.3%})"
for c in cols:
    col = t[c]
    mm = pc.min_max(col) if not pa.types.is_string(col.type) and not pa.types.is_large_string(col.type) else None
    print(f"{c:22s} null={col.null_count / n:7.3%}  " + (f"min={mm['min'].as_py()}  max={mm['max'].as_py()}" if mm else ""))

hvfhs_license_num      null= 0.000%  
request_datetime       null= 0.000%  min=2026-06-29 14:40:33  max=2026-08-01 00:30:00
on_scene_datetime      null= 0.000%  min=2026-06-30 23:09:00  max=2026-07-31 23:59:58
pickup_datetime        null= 0.000%  min=2026-07-01 00:00:00  max=2026-07-31 23:59:59
dropoff_datetime       null= 0.000%  min=2026-07-01 00:02:11  max=2026-08-01 02:04:28
PULocationID           null= 0.000%  min=1  max=265
DOLocationID           null= 0.000%  min=1  max=265
trip_miles             null= 0.000%  min=0.0  max=1431.47
trip_time              null= 0.000%  min=0  max=49332
base_passenger_fare    null= 0.000%  min=-225.21  max=1418.32


## 2. Date range and month coverage
Pickups all fall inside July. Requests start on **29 June**: some rides were requested days before pickup.
That looks like pre-booked rides, and we come back to it in section 5.

In [4]:
days = pc.value_counts(pc.day(t["pickup_datetime"])).to_pylist()
print("distinct pickup days:", len(days), "| rides/day min-max:", min(d["counts"] for d in days), "-", max(d["counts"] for d in days))
print(pc.value_counts(t["hvfhs_license_num"]).to_pylist(), "(HV0003 = Uber, HV0005 = Lyft per TLC data dictionary)")

distinct pickup days: 31 | rides/day min-max: 580100 - 812092
[{'values': 'HV0003', 'counts': 15215374}, {'values': 'HV0005', 'counts': 5705875}] (HV0003 = Uber, HV0005 = Lyft per TLC data dictionary)


## 3. Lifecycle timestamp coverage
The FlashEats lesson was that the "driver arrived at restaurant" event was missing. Here the equivalent is
`on_scene_datetime` (driver arrived at pickup point). Is it present?

In [5]:
for c in ["request_datetime", "on_scene_datetime", "pickup_datetime", "dropoff_datetime"]:
    print(f"% {c:18s} present: {1 - t[c].null_count / n:.3%}")

% request_datetime   present: 100.000%
% on_scene_datetime  present: 100.000%
% pickup_datetime    present: 100.000%
% dropoff_datetime   present: 100.000%


Every timestamp is 100% populated. **But populated is not the same as observed.** Test what the on-scene
values actually mean before trusting them.

In [6]:
r, o, p, d = (t[c] for c in ["request_datetime", "on_scene_datetime", "pickup_datetime", "dropoff_datetime"])
for lic in ["HV0003", "HV0005"]:
    m = pc.equal(t["hvfhs_license_num"], lic)
    k = pc.sum(pc.cast(m, pa.int64())).as_py()
    f = lambda x: f"{pc.sum(pc.cast(pc.and_(m, x), pa.int64())).as_py() / k:.2%}"
    print(lic, f"rides={k:,}")
    print("   on_scene == pickup (to the second):", f(pc.equal(o, p)))
    print("   on_scene <  request               :", f(pc.less(o, r)))
    print("   on_scene >  pickup                :", f(pc.greater(o, p)))

HV0003 rides=15,215,374
   on_scene == pickup (to the second): 6.60%
   on_scene <  request               : 1.77%
   on_scene >  pickup                : 0.00%
HV0005 rides=5,705,875
   on_scene == pickup (to the second): 0.14%
   on_scene <  request               : 1.70%
   on_scene >  pickup                : 0.00%


**Finding:** for HV0003, ~6.6% of rides have on-scene **identical to pickup to the second**. That is not a
plausible physical event; the arrival was probably not recorded separately. Another ~1.8% have on-scene
*before* the request.

**Decision (documented in `validation_report.json`):**
* The headline KPI is **request -> pickup** (authoritative, observed, 100% present). We do **not** call it
  "driver wait time".
* On-scene is used only to split the wait into two stages, and only on rides where
  `request <= on_scene < pickup` (~93% of rides; threshold 80%). No on-scene values are invented.

## 4. Chronology: request <= on_scene <= pickup <= dropoff

In [7]:
def minutes(a, b): return pc.divide(pc.cast(pc.subtract(b, a), pa.int64()), 60e6)
r2p = minutes(r, p)
print("pickup before request :", pct(pc.less(p, r)))
print("dropoff <= pickup     :", pct(pc.less_equal(d, p)))
print("request->pickup quantiles (min):", [round(x, 2) for x in pc.quantile(r2p, q=[0.001, 0.1, 0.5, 0.9, 0.99, 0.999]).to_pylist()])

pickup before request : 246,534 (1.178%)
dropoff <= pickup     : 2 (0.000%)
request->pickup quantiles (min): [-17.02, 1.97, 4.42, 9.98, 18.63, 29.68]


## 5. Why would pickup precede request? Test the pre-scheduled-ride hypothesis

In [8]:
whole = pc.and_(pc.equal(pc.second(r), 0), pc.equal(pc.microsecond(r), 0))
neg = pc.less(p, r)
cnt = lambda x: pc.sum(pc.cast(x, pa.int64())).as_py()
print(f"request on an exact minute | pickup-before-request rows: {cnt(pc.and_(neg, whole)) / cnt(neg):.1%}")
print(f"request on an exact minute | all other rows            : {cnt(pc.and_(pc.invert(neg), whole)) / cnt(pc.invert(neg)):.2%}")
print("chance rate (1/60)                                    : 1.67%")

request on an exact minute | pickup-before-request rows: 82.0%
request on an exact minute | all other rows            : 2.19%
chance rate (1/60)                                    : 1.67%


**Finding:** 82% of the out-of-order rows carry a request stamped on an exact minute, against a 1.7% chance
rate. That is consistent with **pre-scheduled rides storing the booked slot** as `request_datetime`.
The source has no scheduled flag. So:
* the out-of-order rows are **flagged** (`validation_status = pickup_before_request`) and excluded from the KPI
  population (1.2%, below the 5% limit), **not deleted**;
* some scheduled rides probably remain in the population with positive times. That is recorded as **UNKNOWN**,
  and its effect is bounded by a sensitivity run (median barely moves: 4.45 -> 4.47 min).

## 6. Location validity against the Taxi Zone Lookup

In [9]:
import csv
zones = {int(row["LocationID"]): row for row in csv.DictReader(open(ROOT / "data/reference/taxi_zone_lookup.csv", encoding="utf-8-sig"))}
print("lookup rows:", len(zones), "| placeholders:", {k: (v["Borough"], v["Zone"]) for k, v in zones.items() if k >= 264})
known = pa.array(list(zones))
for c in ["PULocationID", "DOLocationID"]:
    print(c, "not in lookup:", pct(pc.invert(pc.is_in(t[c], known))), "| placeholder 264/265:", pct(pc.is_in(t[c], pa.array([264, 265]))))

lookup rows: 265 | placeholders: {264: ('Unknown', 'N/A'), 265: ('N/A', 'Outside of NYC')}
PULocationID not in lookup: 0 (0.000%) | placeholder 264/265: 1,245 (0.006%)
DOLocationID not in lookup: 0 (0.000%) | placeholder 264/265: 978,358 (4.676%)


All IDs resolve. 264/265 are placeholders ("Unknown" / "Outside of NYC"), not zones. That hardly matters for
pickups (0.006%), but 4.7% of dropoffs leave NYC. Placeholder pickups are excluded from the **zone** KPI only.

## 7. Uniqueness and numeric plausibility

In [10]:
import sys; sys.path.insert(0, str(ROOT))
from src.validate import count_natural_key_duplicates  # same exact method the pipeline uses
print("natural-key duplicates:", count_natural_key_duplicates(t))
print("trip_miles <= 0      :", pct(pc.less_equal(t["trip_miles"], 0)))
print("trip_miles > 100     :", pct(pc.greater(t["trip_miles"], 100)))
print("base fare < 0        :", pct(pc.less(t["base_passenger_fare"], 0)))
dur = pc.divide(pc.cast(pc.subtract(d, p), pa.int64()), 1e6)
mism = pc.greater(pc.abs(pc.subtract(dur, pc.cast(t["trip_time"], pa.float64()))), 60)
for lic in ["HV0003", "HV0005"]:
    m = pc.equal(t["hvfhs_license_num"], lic)
    print(f"trip_time vs (dropoff-pickup) off by >60s, {lic}: {cnt(pc.and_(m, mism)) / cnt(m):.2%}")

natural-key duplicates: 0
trip_miles <= 0      : 1,987 (0.009%)
trip_miles > 100     : 3,591 (0.017%)
base fare < 0        : 4,326 (0.021%)
trip_time vs (dropoff-pickup) off by >60s, HV0003: 0.00%
trip_time vs (dropoff-pickup) off by >60s, HV0005: 10.90%


**Finding:** for HV0005, `trip_time` disagrees with the timestamps on ~11% of rides. It is a technically valid
column that is semantically unreliable. **Decision:** durations are derived from timestamps; `trip_time` is not used.

## 8. Weather source: coverage and what the coordinate really is

In [11]:
wdir = ROOT / "data" / "raw" / "weather" / MONTH
wman = json.loads((wdir / "_retrieval_manifest.json").read_text())
w = json.loads((wdir / wman["file"]).read_text())
h = w["hourly"]
print("requested: 40.7812, -73.9665 (Central Park) | returned grid cell:", w["latitude"], w["longitude"], "| tz:", w["timezone"])
print("hours:", len(h["time"]), h["time"][0], "->", h["time"][-1], "| nulls:", {k: sum(v is None for v in vals) for k, vals in h.items()})
pr = h["precipitation"]
print("hours dry:", sum(x == 0 for x in pr), "| light (<2.5 mm):", sum(0 < x < 2.5 for x in pr), "| moderate+:", sum(x >= 2.5 for x in pr))

requested: 40.7812, -73.9665 (Central Park) | returned grid cell: 40.808434 -74.0199 | tz: America/New_York
hours: 768 2026-06-30T00:00 -> 2026-07-31T23:00 | nulls: {'time': 0, 'temperature_2m': 0, 'precipitation': 0, 'wind_speed_10m': 0}
hours dry: 590 | light (<2.5 mm): 146 | moderate+: 32


Open-Meteo snaps the request to a model grid cell about 5 km away. One point stands in for the whole city. Weather
is **context** for an association analysis, joined on the request hour. It is not per-ride weather, and it is
not a cause.

## 9. NYC 311 FHV complaints: is there a customer-friction signal, and at what grain?

311 is the only public source of rider-side friction. Before designing a metric, check what the records
actually are, whether they can be linked to rides, and whether they can be placed in taxi zones.

In [12]:
from collections import Counter
m311 = json.loads((ROOT / "data/raw/nyc311" / MONTH / "_retrieval_manifest.json").read_text())
snap = ROOT / "data/raw/nyc311" / MONTH / m311["snapshot_dir"]
rows311 = [r for p in m311["pages"] for r in json.loads((snap / p["file"]).read_bytes())]
print(f"server count={m311['expected_count']} received={len(rows311)} pages={len(m311['pages'])}")
print("fields with a ride/trip identifier:", [k for k in rows311[0] if "ride" in k or "trip" in k] or "none")
print("descriptor:", Counter(r["descriptor"] for r in rows311).most_common())
print("reason (top 8):", Counter(r.get("descriptor_2") for r in rows311).most_common(8))
print(f"has state-plane x/y: {sum(1 for r in rows311 if r.get('x_coordinate_state_plane')) / len(rows311):.1%}")

server count=1373 received=1373 pages=2
fields with a ride/trip identifier: none
descriptor: [('Driver Complaint - Non Passenger', 1114), ('Driver Complaint - Passenger', 177), ('Car Service Company Complaint', 71), ('Driver Report - Non Passenger', 8), ('Equipment Complaint', 3)]
reason (top 8): [('Unsafe Driving', 1113), ('Fare/Tip/Receipt', 80), ('Discourteous', 35), ('Unauthorized Pick-Up Location', 29), ('Refused Pick-Up', 25), ('Unlicensed/Unauthorized', 20), ('Cell Phone Use', 20), ('Denied Request', 18)]
has state-plane x/y: 97.9%


In [13]:
from src.geo import load_zone_shapes, map_points
shapes = load_zone_shapes(ROOT / "data/reference/taxi_zones.zip")
x = np.array([float(r.get("x_coordinate_state_plane") or "nan") for r in rows311])
y = np.array([float(r.get("y_coordinate_state_plane") or "nan") for r in rows311])
zid, amb = map_points(x, y, shapes)
zb = {s.location_id: s.borough.upper() for s in shapes}
comp = [(zb[int(z)], r["borough"]) for z, r in zip(zid, rows311) if z > 0 and r["borough"] != "Unspecified"]
print(f"mapped to a zone: {(zid > 0).sum()}/{len(rows311)}  ambiguous: {amb.sum()}")
print(f"borough agreement (311 field vs mapped zone): {sum(a == b for a, b in comp) / len(comp):.2%}")
passenger = [z for z, r in zip(zid, rows311) if z > 0 and r["descriptor"] in ("Driver Complaint - Passenger", "Car Service Company Complaint")]
per_zone = Counter(passenger)
print(f"passenger complaints mapped: {len(passenger)} across {len(per_zone)} zones; zones with >=5: {sum(c >= 5 for c in per_zone.values())}")

mapped to a zone: 1342/1373  ambiguous: 0
borough agreement (311 field vs mapped zone): 99.93%
passenger complaints mapped: 237 across 100 zones; zones with >=5: 6


**Findings:**
* There is **no ride identifier**, so complaints can never be joined to rides. Any ride-level join would be invented.
* 81% are *Driver Complaint - Non Passenger* (mostly unsafe driving reported by pedestrians and other road users).
  That is street safety, not rider friction. Only passenger and company complaints count as customer friction.
* The TLC shapefile and 311's state-plane x/y share EPSG:2263. Point-in-polygon maps 97.7% of complaints, and
  **99.9% land in the borough 311 itself recorded**, which independently confirms the mapping.
* Passenger complaints are **sparse**: a few per zone per month. **Decision:** KPI 5 is a borough-month rate per
  100k rides. Zone counts are published only as low-count supporting detail.

## 10. Which metrics are defensible?

| Candidate | Decision | Why |
|---|---|---|
| Median / P90 request->pickup | **KPI 1, 2** | observed timestamps, 100% present, 98.8% chronologically valid |
| Driver-arrival share of the wait | **KPI 3, on the observed subset** | on-scene independently observed for ~93%; subset stated |
| "Driver pickup wait" = pickup - on_scene as headline | **Rejected** | that is curb time; mislabelling it would mislead |
| P90 by pickup zone | **KPI 4, with min-sample rule** | all IDs resolve; tiny zones would be noise |
| Passenger 311 complaints per 100k rides | **KPI 5, borough grain** | real friction signal, but too sparse for zone ranking; never ride-linked |
| Wait by precipitation | **Supporting only** | hour-matched difference ~0.1 min: no operational signal |
| Wait by company | **Supporting only** | informative, but not the operational question |
| Unserved / cancelled requests | **Not possible** | completed trips only |
| Effect of interventions | **Not possible** | proprietary; not in public data |

These conclusions became the KPI definitions in `src/metrics.py` and the rules in `src/validate.py` and `src/nyc311.py`.